<a href="https://colab.research.google.com/github/mdzikrim/MachineLearningClass/blob/main/Chapter11_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
np.random.seed(42)

In [16]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.cifar10.load_data()
y_train_full = y_train_full.squeeze()
y_test = y_test.squeeze()

# Train/valid split
X_valid, y_valid = X_train_full[:5000], y_train_full[:5000]
X_train, y_train = X_train_full[5000:], y_train_full[5000:]

# Common: scale to [0, 1]
X_train_01 = X_train.astype(np.float32) / 255.0
X_valid_01 = X_valid.astype(np.float32) / 255.0
X_test_01  = X_test.astype(np.float32) / 255.0

# For SELU self-normalizing net: standardize inputs (mean 0, std 1)
mean = X_train_01.mean(axis=(0,1,2), keepdims=True)
std  = X_train_01.std(axis=(0,1,2), keepdims=True) + 1e-7

X_train_std = (X_train_01 - mean) / std
X_valid_std = (X_valid_01 - mean) / std
X_test_std  = (X_test_01  - mean) / std

In [17]:
class OneCycleScheduler(keras.callbacks.Callback):
    def __init__(self, max_lr, total_steps, pct_start=0.3, div_factor=10.0, final_div_factor=1e4):
        super().__init__()
        self.max_lr = float(max_lr)
        self.total_steps = int(total_steps)
        self.pct_start = float(pct_start)
        self.div_factor = float(div_factor)
        self.final_div_factor = float(final_div_factor)

        self.initial_lr = self.max_lr / self.div_factor
        self.final_lr = self.initial_lr / self.final_div_factor

    def on_train_begin(self, logs=None):
        self.step = 0
        # Use .assign() for robust learning rate update
        self.model.optimizer.learning_rate.assign(self.initial_lr)

    def _lr_at_step(self, step):
        # Linear up then linear down (simple & robust)
        up_steps = int(self.total_steps * self.pct_start)
        if step < up_steps:
            # increase initial_lr -> max_lr
            ratio = step / max(1, up_steps)
            return self.initial_lr + ratio * (self.max_lr - self.initial_lr)
        else:
            # decrease max_lr -> final_lr
            down_steps = self.total_steps - up_steps
            ratio = (step - up_steps) / max(1, down_steps)
            return self.max_lr + ratio * (self.final_lr - self.max_lr)

    def on_train_batch_begin(self, batch, logs=None):
        lr = self._lr_at_step(self.step)
        # Use .assign() for robust learning rate update
        self.model.optimizer.learning_rate.assign(lr)
        self.step += 1

In [18]:
def build_deep_dnn_elu_he(use_batchnorm=False, dropout_rate=None):
    # He init cocok untuk ReLU & variannya termasuk ELU. :contentReference[oaicite:15]{index=15}
    he_init = keras.initializers.HeNormal()
    model = keras.Sequential()
    model.add(keras.layers.Flatten(input_shape=[32, 32, 3]))

    for _ in range(20):
        if use_batchnorm:
            model.add(keras.layers.Dense(100, kernel_initializer=he_init, use_bias=False))
            model.add(keras.layers.BatchNormalization())
            model.add(keras.layers.Activation("elu"))
        else:
            model.add(keras.layers.Dense(100, activation="elu", kernel_initializer=he_init))
        if dropout_rate is not None:
            model.add(keras.layers.Dropout(dropout_rate))

    model.add(keras.layers.Dense(10, activation="softmax"))
    return model

def build_self_normalizing_selu(alpha_dropout_rate=None):
    # Untuk self-normalizing net: LeCun init + SELU, input distandardisasi, dense stack.
    lecun_init = keras.initializers.LecunNormal()
    model = keras.Sequential()
    model.add(keras.layers.Flatten(input_shape=[32, 32, 3]))

    for _ in range(20):
        model.add(keras.layers.Dense(100, activation="selu", kernel_initializer=lecun_init))
        if alpha_dropout_rate is not None:
            model.add(keras.layers.AlphaDropout(alpha_dropout_rate))

    model.add(keras.layers.Dense(10, activation="softmax"))
    return model

In [19]:
def compile_and_train(model, Xtr, ytr, Xva, yva, *,
                      max_lr=1e-3,
                      batch_size=128,
                      epochs=50,
                      use_onecycle=True):
    optimizer = keras.optimizers.Nadam(learning_rate=max_lr)  # exercise meminta Nadam
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=optimizer,
        metrics=["accuracy"]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    ]

    if use_onecycle:
        steps_per_epoch = int(np.ceil(len(Xtr) / batch_size))
        total_steps = steps_per_epoch * epochs
        callbacks.append(OneCycleScheduler(max_lr=max_lr, total_steps=total_steps))

    history = model.fit(
        Xtr, ytr,
        validation_data=(Xva, yva),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=2
    )
    return history


In [20]:
model_base = build_deep_dnn_elu_he(use_batchnorm=False)
hist_base = compile_and_train(model_base, X_train_01, y_train, X_valid_01, y_valid, max_lr=1e-3)

test_loss, test_acc = model_base.evaluate(X_test_01, y_test, verbose=0)
print("BASE (ELU+He) test acc:", test_acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
352/352 - 24s - 68ms/step - accuracy: 0.3038 - loss: 1.9208 - val_accuracy: 0.3530 - val_loss: 1.8001
Epoch 2/50
352/352 - 8s - 21ms/step - accuracy: 0.3880 - loss: 1.7052 - val_accuracy: 0.3904 - val_loss: 1.7246
Epoch 3/50
352/352 - 11s - 30ms/step - accuracy: 0.4213 - loss: 1.6185 - val_accuracy: 0.4186 - val_loss: 1.6527
Epoch 4/50
352/352 - 11s - 32ms/step - accuracy: 0.4409 - loss: 1.5641 - val_accuracy: 0.4232 - val_loss: 1.6362
Epoch 5/50
352/352 - 9s - 25ms/step - accuracy: 0.4552 - loss: 1.5265 - val_accuracy: 0.4378 - val_loss: 1.6030
Epoch 6/50
352/352 - 7s - 21ms/step - accuracy: 0.4661 - loss: 1.4944 - val_accuracy: 0.4422 - val_loss: 1.6184
Epoch 7/50
352/352 - 19s - 54ms/step - accuracy: 0.4749 - loss: 1.4702 - val_accuracy: 0.4236 - val_loss: 1.6938
Epoch 8/50
352/352 - 8s - 23ms/step - accuracy: 0.4806 - loss: 1.4532 - val_accuracy: 0.4196 - val_loss: 1.7047
Epoch 9/50
352/352 - 8s - 24ms/step - accuracy: 0.4899 - loss: 1.4371 - val_accuracy: 0.4170 - val_l

In [21]:
model_bn = build_deep_dnn_elu_he(use_batchnorm=True)
hist_bn = compile_and_train(model_bn, X_train_01, y_train, X_valid_01, y_valid, max_lr=1e-3)

test_loss_bn, test_acc_bn = model_bn.evaluate(X_test_01, y_test, verbose=0)
print("BN (ELU+He) test acc:", test_acc_bn)

Epoch 1/50
352/352 - 33s - 93ms/step - accuracy: 0.3072 - loss: 1.9577 - val_accuracy: 0.3564 - val_loss: 1.8065
Epoch 2/50
352/352 - 18s - 52ms/step - accuracy: 0.4171 - loss: 1.6350 - val_accuracy: 0.3676 - val_loss: 1.7482
Epoch 3/50
352/352 - 11s - 31ms/step - accuracy: 0.4633 - loss: 1.5136 - val_accuracy: 0.3808 - val_loss: 1.7368
Epoch 4/50
352/352 - 11s - 31ms/step - accuracy: 0.4939 - loss: 1.4337 - val_accuracy: 0.3880 - val_loss: 1.7372
Epoch 5/50
352/352 - 10s - 29ms/step - accuracy: 0.5158 - loss: 1.3717 - val_accuracy: 0.3612 - val_loss: 1.8515
Epoch 6/50
352/352 - 11s - 30ms/step - accuracy: 0.5378 - loss: 1.3180 - val_accuracy: 0.2930 - val_loss: 2.2179
Epoch 7/50
352/352 - 11s - 30ms/step - accuracy: 0.5539 - loss: 1.2735 - val_accuracy: 0.2960 - val_loss: 2.2925
Epoch 8/50
352/352 - 10s - 27ms/step - accuracy: 0.5644 - loss: 1.2379 - val_accuracy: 0.3348 - val_loss: 2.0787
Epoch 9/50
352/352 - 11s - 33ms/step - accuracy: 0.5772 - loss: 1.2011 - val_accuracy: 0.3912 - 

In [22]:
model_selu = build_self_normalizing_selu(alpha_dropout_rate=None)
hist_selu = compile_and_train(model_selu, X_train_std, y_train, X_valid_std, y_valid, max_lr=1e-3)

test_loss_selu, test_acc_selu = model_selu.evaluate(X_test_std, y_test, verbose=0)
print("SELU (self-normalizing) test acc:", test_acc_selu)

Epoch 1/50
352/352 - 18s - 51ms/step - accuracy: 0.3346 - loss: 1.8672 - val_accuracy: 0.3978 - val_loss: 1.6923
Epoch 2/50
352/352 - 9s - 25ms/step - accuracy: 0.4117 - loss: 1.6501 - val_accuracy: 0.4360 - val_loss: 1.6010
Epoch 3/50
352/352 - 9s - 26ms/step - accuracy: 0.4475 - loss: 1.5566 - val_accuracy: 0.4468 - val_loss: 1.5574
Epoch 4/50
352/352 - 8s - 22ms/step - accuracy: 0.4686 - loss: 1.4966 - val_accuracy: 0.4524 - val_loss: 1.5460
Epoch 5/50
352/352 - 8s - 23ms/step - accuracy: 0.4863 - loss: 1.4494 - val_accuracy: 0.4686 - val_loss: 1.5253
Epoch 6/50
352/352 - 8s - 21ms/step - accuracy: 0.5015 - loss: 1.4079 - val_accuracy: 0.4630 - val_loss: 1.5411
Epoch 7/50
352/352 - 9s - 25ms/step - accuracy: 0.5143 - loss: 1.3745 - val_accuracy: 0.4550 - val_loss: 1.5333
Epoch 8/50
352/352 - 8s - 23ms/step - accuracy: 0.5261 - loss: 1.3407 - val_accuracy: 0.4742 - val_loss: 1.5263
Epoch 9/50
352/352 - 8s - 21ms/step - accuracy: 0.5332 - loss: 1.3201 - val_accuracy: 0.4694 - val_loss

In [23]:
model_selu_ad = build_self_normalizing_selu(alpha_dropout_rate=0.1)
hist_selu_ad = compile_and_train(model_selu_ad, X_train_std, y_train, X_valid_std, y_valid, max_lr=1e-3)

# Standard inference
test_loss_ad, test_acc_ad = model_selu_ad.evaluate(X_test_std, y_test, verbose=0)
print("SELU+AlphaDropout standard inference test acc:", test_acc_ad)

Epoch 1/50
352/352 - 23s - 66ms/step - accuracy: 0.1342 - loss: 2.6157 - val_accuracy: 0.1844 - val_loss: 2.6273
Epoch 2/50
352/352 - 9s - 25ms/step - accuracy: 0.1993 - loss: 2.2311 - val_accuracy: 0.2114 - val_loss: 2.8634
Epoch 3/50
352/352 - 10s - 28ms/step - accuracy: 0.2157 - loss: 2.0787 - val_accuracy: 0.1826 - val_loss: 4.3551
Epoch 4/50
352/352 - 10s - 28ms/step - accuracy: 0.2334 - loss: 1.9943 - val_accuracy: 0.1844 - val_loss: 5.4894
Epoch 5/50
352/352 - 10s - 28ms/step - accuracy: 0.2486 - loss: 1.9471 - val_accuracy: 0.1764 - val_loss: 6.9527
Epoch 6/50
352/352 - 9s - 26ms/step - accuracy: 0.2590 - loss: 1.9260 - val_accuracy: 0.1990 - val_loss: 6.3764
Epoch 7/50
352/352 - 10s - 29ms/step - accuracy: 0.2749 - loss: 1.8962 - val_accuracy: 0.2358 - val_loss: 6.1904
Epoch 8/50
352/352 - 10s - 28ms/step - accuracy: 0.2790 - loss: 1.8829 - val_accuracy: 0.2200 - val_loss: 9.0093
Epoch 9/50
352/352 - 10s - 29ms/step - accuracy: 0.3042 - loss: 1.8304 - val_accuracy: 0.2286 - va

In [24]:
@tf.function
def mc_forward_pass(model, X):
    return model(X, training=True)

def mc_dropout_predict_proba(model, X, n_iters=20, batch_size=256):
    # batched loop agar tidak boros memori
    X = tf.convert_to_tensor(X, dtype=tf.float32)
    probs = []
    for _ in range(n_iters):
        probs.append(model(X, training=True))
    probs = tf.stack(probs, axis=0)               # [n_iters, N, n_classes]
    mean_proba = tf.reduce_mean(probs, axis=0)    # [N, n_classes]
    return mean_proba.numpy()

proba_mc = mc_dropout_predict_proba(model_selu_ad, X_test_std, n_iters=20)
y_pred_mc = np.argmax(proba_mc, axis=1)
mc_acc = np.mean(y_pred_mc == y_test)
print("SELU+AlphaDropout MC Dropout test acc:", mc_acc)

SELU+AlphaDropout MC Dropout test acc: 0.2162
